# XAI Lab: Visual explanations


Complete the coding exercises below.

The dataset we will be using is **DermaMNIST**, a subset of the MedMNIST benchmark derived from the HAM10000 collection of dermatoscopic images. It contains skin lesion images across 7 diagnostic categories (e.g., melanoma, nevus, basal cell carcinoma). Each image is a 3-channel RGB photo of a skin lesion, resized to 28×28 pixels in the original benchmark (we upscale to 128×128 here). We will train a CNN classifier on this dataset and apply various saliency-based attribution methods to explain its predictions.

In [2]:
!pip install medmnist
!pip install captum --no-deps
!pip install opencv

ERROR: Could not find a version that satisfies the requirement opencv (from versions: none)
ERROR: No matching distribution found for opencv


In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F

import medmnist
from medmnist import INFO
from medmnist.dataset import DermaMNIST

from captum.attr import Saliency, NoiseTunnel, LayerGradCam, LayerAttribution, LRP
import cv2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
data_flag = 'dermamnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

# Transform (resize - optional)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Load datasets
train_dataset = DataClass(split='train', transform=transform, download=True, size=128)
val_dataset   = DataClass(split='val',   transform=transform, download=True, size=128)
test_dataset  = DataClass(split='test',  transform=transform, download=True, size=128)

# Build loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)


100%|██████████| 373M/373M [07:39<00:00, 812kB/s] 


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
            )

        self.conv1 = block(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = block(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = block(64, 128)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x_gap = self.gap(x).view(x.size(0), -1)
        out = self.fc(x_gap)
        return out

model = SimpleCNN(num_classes=7).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, lbls = imgs.to(device), lbls.squeeze().long().to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Train Loss: {avg_loss:.4f}")

In [ ]:
model.eval()

seed = 43
torch.manual_seed(seed)

# Pick a random image from validation loader
val_iter = iter(val_loader)
img_batch, label_batch = next(val_iter)
rand_idx = torch.randint(0, img_batch.size(0), (1,)).item()
img, label = img_batch[rand_idx:rand_idx+1].to(device), label_batch[rand_idx].item()

print(f"Selected image index: {rand_idx}, label: {label}")

We have defined a model, trained it on the data for a few epochs and picked one image from the validation set.
- Compute the following attribution maps for the selected image:
  - **Task (1.5 point)** Vanilla Gradients
  - **Task (2 points)** SmoothGrad
  - **Task (2 points)** GradCAM
  - **Task (2 points)** LRP

  (Optional: Try to implement some of the methods above without using Captum).

- **Task (0.5 point)** Visualize the original image and all attribution maps side-by-side in a single figure
- **Task (1 point)** Identify two missclassified samples and compute all 4 attribution maps for each.
- **Task (1 point)** Quantify how much each attrbution method focuses on the object vs background:
  - For a selected image, define a central mask (e.g., central 50% of the image) as the “object region”.
  - For each attribution map compute the fraction of saliency inside the object region

### Coding tasks:

Compute the following attribution maps for the selected image:
- **Task (1.5 point)** Vanilla Gradients

In [ ]:
# Your solution here ...
img.requires_grad = True
saliency = Saliency(model)
pred = model(img).argmax(dim=1).item()
attr_vanilla = saliency.attribute(img, target=pred)

attr_vanilla_np = attr_vanilla.squeeze().cpu().detach().numpy()
attr_vanilla_vis = np.mean(np.abs(attr_vanilla_np), axis=0)

plt.imshow(attr_vanilla_vis, cmap='hot')
plt.title(f"Vanilla Gradients (pred={pred}, true={label})")
plt.colorbar()
plt.show()

- **Task (2 points)** SmoothGrad

In [ ]:
# Your solution here ...
smooth = NoiseTunnel(saliency)
attr_smooth = smooth.attribute(img, nt_type='smoothgrad', nt_samples=50, stdevs=0.2, target=pred)

attr_smooth_np = attr_smooth.squeeze().cpu().detach().numpy()
attr_smooth_vis = np.mean(np.abs(attr_smooth_np), axis=0)

plt.imshow(attr_smooth_vis, cmap='hot')
plt.title(f"SmoothGrad (pred={pred}, true={label})")
plt.colorbar()
plt.show()

- **Task (2 points)** GradCAM

In [ ]:
# Your solution here ...
gradcam = LayerGradCam(model, model.conv3)
attr_gradcam = gradcam.attribute(img, target=pred)

  # Upsample to input size
attr_gradcam_up = LayerAttribution.interpolate(attr_gradcam, (128, 128))
attr_gradcam_vis = attr_gradcam_up.squeeze().cpu().detach().numpy()

plt.imshow(attr_gradcam_vis, cmap='hot')
plt.title(f"GradCAM (pred={pred}, true={label})")
plt.colorbar()
plt.show()

- **Task (2 points)** LRP (below with Captum)

In [ ]:
class SimpleCNN_LRP(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=False),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=False),
            )
        self.conv1 = block(3, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = block(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = block(64, 128)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x_gap = self.gap(x).view(x.size(0), -1)
        out = self.fc(x_gap)
        return out

model_lrp = SimpleCNN_LRP(num_classes=7).to(device)
model_lrp.load_state_dict(model.state_dict())
model_lrp.eval()

lrp = LRP(model_lrp)
attr_lrp = lrp.attribute(img, target=pred)

attr_lrp_np = attr_lrp.squeeze().cpu().detach().numpy()
attr_lrp_vis = np.mean(np.abs(attr_lrp_np), axis=0)

plt.imshow(attr_lrp_vis, cmap='hot')
plt.title(f"LRP (pred={pred}, true={label})")
plt.colorbar()
plt.show()

**Task (0.5 point)** Visualize the original image and all attribution maps side-by-side in a single figure

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

  # Original image (denormalize)
orig = img.squeeze().cpu().detach().numpy().transpose(1, 2, 0)
orig = (orig * 0.5) + 0.5  # undo normalize
axes[0].imshow(np.clip(orig, 0, 1))
axes[0].set_title(f"Original (true={label}, pred={pred})")

axes[1].imshow(attr_vanilla_vis, cmap='hot')
axes[1].set_title("Vanilla Gradients")

axes[2].imshow(attr_smooth_vis, cmap='hot')
axes[2].set_title("SmoothGrad")

axes[3].imshow(attr_gradcam_vis, cmap='hot')
axes[3].set_title("GradCAM")

axes[4].imshow(attr_lrp_vis, cmap='hot')
axes[4].set_title("LRP")

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


**Task (1 point)** Identify two missclassified samples and compute all 4 attribution maps for each.

In [ ]:
# Your solution here ...

**Task (1 point)** Quantify how much each attrbution method focuses on the object vs background

In [ ]:
# Your solution here ...